# 🔧 Lab 04: Embedding Finetune 實作

## 學習目標
在本實驗中，您將學習：
1. **Sentence Transformers 訓練流程** - 完整的 finetune pipeline
2. **Loss Functions 比較** - MultipleNegativesRankingLoss vs CosineSimilarityLoss
3. **Training 配置** - Learning rate, batch size, warmup 等參數
4. **評估與比較** - 量化 finetune 前後的效能差異

## 為什麼需要 Finetune？
- 預訓練模型是通用的，可能不適合特定領域
- Finetune 可以讓模型學習領域專屬的語義
- 通常只需要少量資料就能看到顯著改進

## 技術棧
- **Training Framework**: `sentence-transformers`
- **Hardware**: GPU 推薦 (CPU 也可以，但較慢)

---

## 📦 Part 1: 環境設置

In [ ]:
# 安裝必要套件
!pip install --quiet sentence-transformers>=2.2.0
!pip install --quiet torch
!pip install --quiet pandas numpy matplotlib
!pip install --quiet scikit-learn
!pip install --quiet rank_bm25

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Set, Tuple

import torch
from torch.utils.data import DataLoader

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
)
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator,
    EmbeddingSimilarityEvaluator,
)
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

# 檢查 GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ 使用裝置: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 📊 Part 2: 準備訓練資料

### 建立 FAQ 資料集

我們將建立一個客服 FAQ 資料集，包含：
- 問題 (Queries)
- 標準答案 (Positives)
- 用於挖掘負樣本的文件庫 (Corpus)

In [ ]:
# 建立 FAQ 資料集

faq_data = [
    # (問題, 標準答案)
    ("How do I reset my password?", 
     "To reset your password, go to the login page and click 'Forgot Password'. Enter your email and follow the instructions sent to your inbox."),
    
    ("What are your business hours?", 
     "Our customer service is available Monday to Friday, 9 AM to 6 PM EST. For urgent matters, you can use our 24/7 chatbot."),
    
    ("How can I track my order?", 
     "You can track your order by logging into your account and visiting 'My Orders'. Alternatively, use the tracking number from your confirmation email."),
    
    ("What is your return policy?", 
     "We offer a 30-day return policy for unused items in original packaging. Contact customer support to initiate a return."),
    
    ("How do I cancel my subscription?", 
     "To cancel your subscription, go to Account Settings > Subscription > Cancel. You'll retain access until the end of your billing period."),
    
    ("Do you offer international shipping?", 
     "Yes, we ship to over 100 countries. International shipping rates and delivery times vary by location. Check our shipping page for details."),
    
    ("How can I contact customer support?", 
     "You can reach customer support via email at support@example.com, phone at 1-800-123-4567, or live chat on our website."),
    
    ("What payment methods do you accept?", 
     "We accept Visa, MasterCard, American Express, PayPal, Apple Pay, and Google Pay. All transactions are secured with SSL encryption."),
    
    ("How do I update my billing information?", 
     "Go to Account Settings > Payment Methods. You can add, edit, or remove payment methods. Changes take effect immediately."),
    
    ("Is my personal data secure?", 
     "Yes, we use industry-standard encryption and security protocols. Your data is stored securely and never shared with third parties without consent."),
    
    ("How do I change my email address?", 
     "Go to Account Settings > Personal Information > Email. Enter your new email and verify it by clicking the link sent to your new address."),
    
    ("Can I get a refund?", 
     "Refunds are processed within 5-7 business days after we receive your returned item. The refund will be credited to your original payment method."),
    
    ("How do I apply a discount code?", 
     "Enter your discount code in the 'Promo Code' field at checkout and click 'Apply'. The discount will be reflected in your order total."),
    
    ("What if my package is lost?", 
     "If your package is lost, contact us with your order number. We'll investigate with the carrier and either resend the items or issue a full refund."),
    
    ("How do I create an account?", 
     "Click 'Sign Up' on our homepage. Enter your email, create a password, and complete the registration form. You'll receive a confirmation email."),
]

# 分割為訓練和測試集
train_data = faq_data[:12]
test_data = faq_data[12:]

print(f"📊 資料集統計:")
print(f"   訓練集: {len(train_data)} 個問答對")
print(f"   測試集: {len(test_data)} 個問答對")

In [ ]:
# 建立文件庫 (用於 hard negative mining 和評估)
corpus = [answer for _, answer in faq_data]

# 加入一些干擾文件
additional_docs = [
    "Our company was founded in 2010 and has grown to serve millions of customers worldwide.",
    "Subscribe to our newsletter to receive exclusive offers and updates about new products.",
    "Our mobile app is available on both iOS and Android platforms for convenient shopping.",
    "We value customer feedback and continuously improve our services based on your suggestions.",
    "Join our loyalty program to earn points on every purchase and unlock special rewards.",
]
corpus.extend(additional_docs)

print(f"📚 文件庫: {len(corpus)} 個文件")

In [ ]:
# Hard Negative Mining
from rank_bm25 import BM25Okapi

def mine_hard_negatives(
    query: str,
    positive: str,
    corpus: List[str],
    model: SentenceTransformer,
    top_k: int = 3
) -> List[str]:
    """使用語義相似度挖掘 hard negatives"""
    q_emb = model.encode(query)
    c_embs = model.encode(corpus)
    
    sims = cosine_similarity([q_emb], c_embs)[0]
    sorted_indices = np.argsort(sims)[::-1]
    
    negatives = []
    for idx in sorted_indices:
        if corpus[idx] != positive and len(negatives) < top_k:
            negatives.append(corpus[idx])
    
    return negatives

# 載入基礎模型用於 mining
base_model = SentenceTransformer('all-MiniLM-L6-v2')

# 為訓練資料建立完整的三元組
training_triplets = []
for query, positive in train_data:
    negatives = mine_hard_negatives(query, positive, corpus, base_model, top_k=2)
    for negative in negatives:
        training_triplets.append((query, positive, negative))

print(f"✅ 建立了 {len(training_triplets)} 個訓練三元組")
print(f"\n📝 範例:")
print(f"   Query: {training_triplets[0][0]}")
print(f"   Positive: {training_triplets[0][1][:60]}...")
print(f"   Negative: {training_triplets[0][2][:60]}...")

---
## 🔢 Part 3: 建立 InputExamples

Sentence Transformers 使用 `InputExample` 類別來封裝訓練資料。

In [ ]:
# 方法 1: 使用 Pairs (用於 MultipleNegativesRankingLoss)
def create_pair_examples(data: List[Tuple[str, str]]) -> List[InputExample]:
    """建立 (query, positive) pair 格式的 InputExamples"""
    examples = []
    for query, positive in data:
        examples.append(InputExample(texts=[query, positive]))
    return examples

# 方法 2: 使用 Triplets (用於 TripletLoss)
def create_triplet_examples(triplets: List[Tuple[str, str, str]]) -> List[InputExample]:
    """建立 (query, positive, negative) triplet 格式的 InputExamples"""
    examples = []
    for query, positive, negative in triplets:
        examples.append(InputExample(texts=[query, positive, negative]))
    return examples

# 建立訓練資料
train_examples_pairs = create_pair_examples(train_data)
train_examples_triplets = create_triplet_examples(training_triplets)

print(f"Pair examples: {len(train_examples_pairs)}")
print(f"Triplet examples: {len(train_examples_triplets)}")

---
## 📉 Part 4: Loss Functions 介紹

### 常用 Loss Functions

| Loss Function | 資料格式 | 說明 |
|--------------|---------|------|
| `MultipleNegativesRankingLoss` | Pairs | 使用 in-batch negatives，效率高 |
| `TripletLoss` | Triplets | 明確的三元組，直接優化 |
| `CosineSimilarityLoss` | Pairs + Labels | 適合有標籤的相似度資料 |
| `ContrastiveLoss` | Pairs + Labels | 經典對比學習 loss |

In [ ]:
# MultipleNegativesRankingLoss 說明
print("📖 MultipleNegativesRankingLoss")
print("=" * 60)
print()
print("特點:")
print("- 只需要 (query, positive) pairs")
print("- 自動使用同 batch 中其他樣本的 positive 作為 negatives")
print("- 更大的 batch size = 更多 negatives = 更好的效果")
print()
print("範例 (batch size = 3):")
print("  Batch: [(q1, p1), (q2, p2), (q3, p3)]")
print("  對於 q1: positive=p1, negatives=[p2, p3]")
print("  對於 q2: positive=p2, negatives=[p1, p3]")
print("  對於 q3: positive=p3, negatives=[p1, p2]")
print()
print("💡 推薦用於 retrieval 任務，效率高！")

---
## 📊 Part 5: 建立評估器

在訓練過程中監控模型效能，確保模型在改進。

In [ ]:
# 建立 Information Retrieval Evaluator
def create_ir_evaluator(
    queries: List[Tuple[str, str]],
    corpus: List[str],
    name: str = "faq_eval"
) -> InformationRetrievalEvaluator:
    """
    建立 IR Evaluator
    
    Args:
        queries: (query, answer) 對列表
        corpus: 文件庫
        name: 評估器名稱
    
    Returns:
        InformationRetrievalEvaluator
    """
    # 建立 queries dict: {query_id: query_text}
    queries_dict = {f"q{i}": q for i, (q, _) in enumerate(queries)}
    
    # 建立 corpus dict: {corpus_id: doc_text}
    corpus_dict = {f"c{i}": doc for i, doc in enumerate(corpus)}
    
    # 建立 relevant_docs: {query_id: {corpus_id}}
    # 找出每個 query 對應的 answer 在 corpus 中的位置
    relevant_docs = {}
    for i, (q, a) in enumerate(queries):
        query_id = f"q{i}"
        for j, doc in enumerate(corpus):
            if doc == a:
                relevant_docs[query_id] = {f"c{j}"}
                break
    
    evaluator = InformationRetrievalEvaluator(
        queries=queries_dict,
        corpus=corpus_dict,
        relevant_docs=relevant_docs,
        name=name,
        mrr_at_k=[1, 3, 5, 10],
        ndcg_at_k=[1, 3, 5, 10],
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
    )
    
    return evaluator

# 建立評估器
dev_evaluator = create_ir_evaluator(test_data, corpus, name="dev")
print("✅ 評估器建立完成")

In [ ]:
# 評估基礎模型 (finetune 前)
print("📊 評估基礎模型 (Finetune 前)...")
baseline_results = dev_evaluator(base_model)

print("\n基礎模型效能:")
print(f"  MRR@10: {baseline_results:.4f}")

---
## 🏋️ Part 6: 訓練模型

### 訓練配置

重要參數：
- **Batch Size**: 越大越好 (受限於 GPU 記憶體)
- **Epochs**: 通常 1-5 epochs 足夠
- **Learning Rate**: 2e-5 是常用起點
- **Warmup Steps**: 讓學習率緩慢上升，避免初期震盪

In [ ]:
# 訓練配置
MODEL_NAME = 'all-MiniLM-L6-v2'
OUTPUT_PATH = './finetuned_model'
BATCH_SIZE = 8  # 根據 GPU 記憶體調整
EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

# 建立輸出目錄
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("🔧 訓練配置:")
print(f"   模型: {MODEL_NAME}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   輸出路徑: {OUTPUT_PATH}")

In [ ]:
# 方法 1: 使用 MultipleNegativesRankingLoss 訓練

print("🚀 開始訓練 (MultipleNegativesRankingLoss)...")
print("=" * 60)

# 載入模型
model_mnr = SentenceTransformer(MODEL_NAME)

# 建立 DataLoader
train_dataloader_mnr = DataLoader(
    train_examples_pairs, 
    shuffle=True, 
    batch_size=BATCH_SIZE
)

# 建立 Loss
train_loss_mnr = losses.MultipleNegativesRankingLoss(model_mnr)

# 計算訓練步數
num_training_steps = len(train_dataloader_mnr) * EPOCHS
warmup_steps = int(num_training_steps * WARMUP_RATIO)

print(f"   訓練樣本: {len(train_examples_pairs)}")
print(f"   總步數: {num_training_steps}")
print(f"   Warmup 步數: {warmup_steps}")
print()

# 訓練
model_mnr.fit(
    train_objectives=[(train_dataloader_mnr, train_loss_mnr)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    evaluator=dev_evaluator,
    evaluation_steps=len(train_dataloader_mnr),  # 每個 epoch 評估一次
    output_path=f"{OUTPUT_PATH}/mnr",
    save_best_model=True,
    show_progress_bar=True,
)

print("\n✅ 訓練完成！")

In [ ]:
# 方法 2: 使用 TripletLoss 訓練

print("🚀 開始訓練 (TripletLoss)...")
print("=" * 60)

# 載入模型
model_triplet = SentenceTransformer(MODEL_NAME)

# 建立 DataLoader
train_dataloader_triplet = DataLoader(
    train_examples_triplets, 
    shuffle=True, 
    batch_size=BATCH_SIZE
)

# 建立 Loss
train_loss_triplet = losses.TripletLoss(model_triplet)

# 計算訓練步數
num_training_steps = len(train_dataloader_triplet) * EPOCHS
warmup_steps = int(num_training_steps * WARMUP_RATIO)

print(f"   訓練樣本: {len(train_examples_triplets)}")
print(f"   總步數: {num_training_steps}")
print(f"   Warmup 步數: {warmup_steps}")
print()

# 訓練
model_triplet.fit(
    train_objectives=[(train_dataloader_triplet, train_loss_triplet)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    evaluator=dev_evaluator,
    evaluation_steps=len(train_dataloader_triplet),
    output_path=f"{OUTPUT_PATH}/triplet",
    save_best_model=True,
    show_progress_bar=True,
)

print("\n✅ 訓練完成！")

---
## 📈 Part 7: 評估與比較

In [ ]:
# 比較三個模型的效能

print("📊 模型比較")
print("=" * 60)

# 載入最佳模型
model_baseline = SentenceTransformer(MODEL_NAME)

# 載入 finetuned 模型
try:
    model_mnr_best = SentenceTransformer(f"{OUTPUT_PATH}/mnr")
    mnr_available = True
except:
    model_mnr_best = model_mnr
    mnr_available = False

try:
    model_triplet_best = SentenceTransformer(f"{OUTPUT_PATH}/triplet")
    triplet_available = True
except:
    model_triplet_best = model_triplet
    triplet_available = False

# 評估
results = {}

print("\n評估基礎模型...")
results['Baseline'] = dev_evaluator(model_baseline)

print("評估 MNR Loss 模型...")
results['MNR Loss'] = dev_evaluator(model_mnr_best)

print("評估 Triplet Loss 模型...")
results['Triplet Loss'] = dev_evaluator(model_triplet_best)

# 顯示結果
print("\n" + "=" * 60)
print("📊 結果比較 (MRR@10)")
print("=" * 60)

for model_name, score in results.items():
    improvement = ((score - results['Baseline']) / results['Baseline']) * 100 if model_name != 'Baseline' else 0
    status = f"(+{improvement:.1f}%)" if improvement > 0 else f"({improvement:.1f}%)"
    print(f"  {model_name}: {score:.4f} {status if model_name != 'Baseline' else ''}")

In [ ]:
# 視覺化比較
models = list(results.keys())
scores = list(results.values())

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = plt.bar(models, scores, color=colors)

# 加上數值標籤
for bar, score in zip(bars, scores):
    plt.text(
        bar.get_x() + bar.get_width()/2, 
        bar.get_height() + 0.01,
        f'{score:.4f}',
        ha='center',
        fontsize=12,
        fontweight='bold'
    )

plt.ylabel('MRR@10', fontsize=12)
plt.title('Model Comparison: Finetune Effect', fontsize=14)
plt.ylim(0, 1.1)

# 加上基準線
plt.axhline(y=results['Baseline'], color='gray', linestyle='--', alpha=0.5, label='Baseline')

plt.tight_layout()
plt.show()

In [ ]:
# 定性評估：查看具體檢索結果

def show_retrieval_results(
    query: str,
    corpus: List[str],
    models: Dict[str, SentenceTransformer],
    top_k: int = 3
):
    """顯示不同模型的檢索結果"""
    print(f"Query: '{query}'")
    print("=" * 70)
    
    for model_name, model in models.items():
        q_emb = model.encode(query)
        c_embs = model.encode(corpus)
        sims = cosine_similarity([q_emb], c_embs)[0]
        
        top_indices = np.argsort(sims)[::-1][:top_k]
        
        print(f"\n{model_name}:")
        for rank, idx in enumerate(top_indices, 1):
            print(f"  {rank}. [{sims[idx]:.3f}] {corpus[idx][:70]}...")

# 測試查詢
test_queries = [
    "I forgot my password",
    "Can I get my money back?",
    "How to use discount",
]

comparison_models = {
    'Baseline': model_baseline,
    'MNR Finetuned': model_mnr_best,
}

for query in test_queries:
    show_retrieval_results(query, corpus, comparison_models)
    print("\n" + "-" * 70 + "\n")

---
## 💾 Part 8: 儲存與載入模型

In [ ]:
# 儲存最佳模型
FINAL_MODEL_PATH = './finetuned_model/best'

# 選擇效能最好的模型儲存
best_model_name = max(results, key=results.get)
print(f"🏆 最佳模型: {best_model_name}")

if best_model_name == 'MNR Loss':
    best_model = model_mnr_best
elif best_model_name == 'Triplet Loss':
    best_model = model_triplet_best
else:
    best_model = model_baseline

# 儲存
best_model.save(FINAL_MODEL_PATH)
print(f"✅ 模型已儲存至: {FINAL_MODEL_PATH}")

In [ ]:
# 載入模型
loaded_model = SentenceTransformer(FINAL_MODEL_PATH)

# 測試載入的模型
test_query = "How do I reset my password?"
embedding = loaded_model.encode(test_query)

print(f"✅ 模型載入成功！")
print(f"   Embedding 維度: {embedding.shape}")

---
## 🏋️ 練習: 在你自己的資料上 Finetune

嘗試使用你自己的領域資料進行 finetune。

In [ ]:
# TODO: 使用你自己的資料進行 finetune

# 1. 準備你的資料
my_data = [
    ("你的問題 1", "對應的答案 1"),
    ("你的問題 2", "對應的答案 2"),
    # 加入更多...
]

# 2. 建立 InputExamples
my_examples = create_pair_examples(my_data)

# 3. 訓練
# my_model = SentenceTransformer('all-MiniLM-L6-v2')
# my_dataloader = DataLoader(my_examples, shuffle=True, batch_size=4)
# my_loss = losses.MultipleNegativesRankingLoss(my_model)
# my_model.fit(...)

print("📝 請修改上面的程式碼，使用你自己的資料進行訓練！")

---
## 📝 總結

### 本實驗學到的重點

1. **Sentence Transformers 訓練流程**
   - 準備 InputExamples
   - 選擇適合的 Loss Function
   - 建立 Evaluator 監控訓練

2. **Loss Functions**
   - MultipleNegativesRankingLoss: 簡單高效，推薦首選
   - TripletLoss: 需要明確的三元組，可控性高

3. **Training 技巧**
   - 較大的 batch size 通常效果更好
   - Warmup 幫助穩定訓練初期
   - 使用 evaluator 監控避免 overfitting

4. **效能評估**
   - 使用 IR Evaluator 量化效能
   - 比較 finetune 前後的改進

### 下一步
在 Lab 05 中，我們將探索 Sparse Representations 和 Seismic！

---
## 📚 參考資源

- [Sentence Transformers Training](https://www.sbert.net/docs/training/overview.html)
- [Loss Functions 詳解](https://www.sbert.net/docs/package_reference/losses.html)
- [Training Examples](https://github.com/UKPLab/sentence-transformers/tree/master/examples/training)
- [Hugging Face Model Hub](https://huggingface.co/models?library=sentence-transformers)